# Create an editable cohort query

## Goal

Define a small query mapping, execute it, and reshape its body-weight and
hematocrit observations into an indexed daily table.

In [ ]:
import os
from pathlib import Path

source_from_environment = os.environ.get("VCGEN_SEND_ARCHIVE")
if source_from_environment:
    SEND_SOURCE = Path(source_from_environment)
else:
    SEND_SOURCE = next(
        (
            root / "rodent_db_export.zip"
            for root in (Path.cwd(), *Path.cwd().parents)
            if (root / "rodent_db_export.zip").is_file()
        ),
        Path("rodent_db_export.zip"),
    )

EXTRACTION_CACHE = Path(
    os.environ.get("VCGEN_QUERY_CACHE", SEND_SOURCE.parent / ".vcgen_query_cache")
)

if not SEND_SOURCE.exists():
    raise FileNotFoundError(
        "Set VCGEN_SEND_ARCHIVE to a SEND ZIP archive or extracted CSV directory; "
        f"got {SEND_SOURCE}"
    )

## 1. Define the query

In [ ]:
import vcgen_query as vq

subject_query_mapping = {
    "contract": "vcgen.subject_query.v1",
    "query_id": "example_rat_body_weight",
    "predicate": {
        "group": "and",
        "children": [
            {
                "field": {
                    "kind": "derived",
                    "name": "planned_dose_duration_days",
                    "value_type": "number",
                },
                "operator": "between",
                "value": [28, 28],
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "DM",
                    "column": "SPECIES",
                    "value_type": "string",
                },
                "operator": "eq",
                "value": "RAT",
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "DM",
                    "column": "SEX",
                    "value_type": "string",
                },
                "operator": "in",
                "value": ["F", "M"],
            },
            {
                "field": {
                    "kind": "send_column",
                    "domain": "EX",
                    "column": "EXROUTE",
                    "value_type": "string",
                },
                "operator": "eq",
                "value": "ORAL GAVAGE",
            },
        ],
    },
}

# The subject query selects the cohort. Endpoint selection is a later stage.
subject_query = vq.parse_subject_query(subject_query_mapping)
query = vq.CohortQuery.compose(subject_query, ["BW", "LB_HCT"])
{
    "query_id": query.query_id,
    "predicate_fingerprint": subject_query.fingerprint,
    "stages": [stage.info.id for stage in query.stages],
}

## 2. Execute and reshape

In [ ]:
result = vq.execute_query(
    SEND_SOURCE,
    query,
    extract_dir=EXTRACTION_CACHE,
)

[
    {"stage": record.info.id, "in": dict(record.n_in), "out": dict(record.n_out)}
    for record in result.trace
]

In [ ]:
KEY_COLUMNS = ["study_id", "subject_id", "study_day"]
daily_values = (
    vq.to_daily_frame(result)
    .values.set_index(KEY_COLUMNS)
    .sort_index()
)

daily_values[["BW", "LB_HCT"]].dropna(how="all").head(12)

## Takeaway

Edit the subject query to change cohort membership. Its output is the Grit42
parity checkpoint. Edit the endpoint stage to change extracted endpoints. The
trace shows the subject and observation counts after each stage.